# 02 · Tratamento Silver (Bônus 1)
**Parte 1 — Analisar:** investigar a tabela bronze em busca de problemas e inconsistências.

**Parte 2 — Tratar:** corrigir os problemas encontrados e gravar `workspace.fauna.silver_registros`.

In [0]:
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")
print("Fuso da sessão:", spark.conf.get("spark.sql.session.timeZone"))

In [0]:
from pyspark.sql import functions as F

bronze = spark.table("workspace.fauna.bronze_registros")
colunas = [c for c in bronze.columns if not c.startswith("_")]

bronze.select(colunas).printSchema()

## Parte 1 — Analisar

In [0]:
display(bronze.limit(10))

In [0]:
nulos = bronze.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in colunas
])
display(nulos)

In [0]:
total = bronze.count()
ids_unicos = bronze.select("id_registro").distinct().count()

sem_id = [c for c in colunas if c != "id_registro"]
linhas_unicas = bronze.dropDuplicates(sem_id).count()

print(f"Total de registros           : {total}")
print(f"IDs repetidos                : {total - ids_unicos}")
print(f"Registros repetidos (sem ID) : {total - linhas_unicas}")

In [0]:
display(
    bronze.groupBy("grupo", "especie")
    .count()
    .orderBy("grupo", F.desc("count"))
)

In [0]:
display(bronze.groupBy("id_camera").count().orderBy("id_camera"))

In [0]:
display(bronze.select(
    F.min("data_hora_inicio").alias("primeiro_registro"),
    F.max("data_hora_inicio").alias("ultimo_registro"),
))

In [0]:
display(bronze.select("duracao_segundos", "individuos").summary())

In [0]:
display(
    bronze.select("especie", "id_camera", "duracao_segundos")
    .orderBy(F.desc("duracao_segundos"))
    .limit(10)
)

### Pontos de Atenção
| # | Problema | Coluna(s) | Impacto nas análises |
|---|---|---|---|
| 1 | Tipo de dados | data_hora_inicio | Verificar a possível necessidade no futuro de separação da data e hora para analises de horários comuns |
| 2 | Registros com "°C" | temperatura_c | Transforma campo em decimal para calculos matemáticos |
| 3 | Registros com "%" | umidade_pct | Transforma campo em decimal para calculos matemáticos |
| 4 | Registros nulos | temperatura_c e umidade_pct | Analisar algum padrão ou definir como atenção para definir analises no futuro |

## Parte 2 — Tratar

In [0]:
area = (
    F.when(F.col("id_camera").isin("CAM01", "CAM02", "CAM03", "CAM04"), "Cerrado aberto")
     .when(F.col("id_camera").isin("CAM05", "CAM06", "CAM07", "CAM08"), "Mata de galeria")
     .when(F.col("id_camera").isin("CAM09", "CAM10", "CAM11", "CAM12"), "Veredas e áreas úmidas")
)

silver = (
    bronze
    .select(
        "id_registro",
        "id_camera",
        area.alias("area"),
        F.col("data_hora_inicio").cast("timestamp_ntz").alias("data_hora_inicio"),
        "duracao_segundos",
        "especie",
        "grupo",
        "individuos",
        # " °C" e "%"
        F.regexp_replace("temperatura_c", "[^0-9.]", "").cast("double").alias("temperatura_c"),
        F.regexp_replace("umidade_pct", "[^0-9.]", "").cast("double").alias("umidade_pct"),
    )
    .withColumn("data", F.to_date("data_hora_inicio"))
    .withColumn("hora", F.hour("data_hora_inicio"))
    .withColumn(
        "periodo_dia",
        F.when(F.col("hora") < 6, "Madrugada")
         .when(F.col("hora") < 12, "Manhã")
         .when(F.col("hora") < 18, "Tarde")
         .otherwise("Noite"),
    )
    .withColumn("_data_processamento", F.current_timestamp())
)

silver.printSchema()
display(silver.limit(10))

In [0]:
def qtd_nulos(df, coluna):
    return df.filter(F.col(coluna).isNull()).count()

checagens = {
    "5000 registros": silver.count() == 5000,
    "12 câmeras": silver.select("id_camera").distinct().count() == 12,
    "13 espécies": silver.select("especie").distinct().count() == 13,
    "3 grupos": silver.select("grupo").distinct().count() == 3,
    "Todas as câmeras tem área": qtd_nulos(silver, "area") == 0,
    "Datas entre 01/05 e 30/05/2026": silver.filter(~F.col("data").between("2026-05-01", "2026-05-30")).count() == 0,
    "Temperatura: nenhum nulo novo na conversão": qtd_nulos(silver, "temperatura_c") == qtd_nulos(bronze, "temperatura_c"),
    "umidade: nenhum nulo novo na conversão": qtd_nulos(silver, "umidade_pct") == qtd_nulos(bronze, "umidade_pct"),
}

for nome, ok in checagens.items():
    print(f"{'OK -' if ok else 'ERRO -'} {nome}")

assert all(checagens.values()), "Alguma checagem falhou: corrija antes de gravar a silver!"

In [0]:
display(silver.select("temperatura_c", "umidade_pct", "hora").summary())

In [0]:
(
    silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.fauna.silver_registros")
)

In [0]:
%sql
DESCRIBE TABLE workspace.fauna.silver_registros

### Decisões de tratamento
| Problema | Decisão | Justificativa |
|---|---|---|
| `data_hora_inicio` como texto | Converter para `TIMESTAMP_NTZ` | Hora de relógio local, sem risco de deslocamento por fuso |
| Temperatura e umidade com unidade (`°C`, `%`) | Remover a unidade e converter para `double` | Permitir cálculos e faixas de valores |
| Nulos em temperatura (247) e umidade (175) | Manter as linhas, sem imputar valores | O restante do registro é válido; os nulos são ignorados só nas análises que usam essas colunas |
| Não há coluna de área | Criar `area` a partir da câmera | Necessária para os Desafios 4 e 5 |
| Não há período do dia | Criar `hora` e `periodo_dia` (Madrugada 0–5h, Manhã 6–11h, Tarde 12–17h, Noite 18–23h) | Necessária para os Desafios 2 e 5 |
| Durações longas (máx. 167 s) | Manter | Valores positivos e plausíveis (< 3 min); serão investigados no Bônus 2 |
| Duplicidades | Nenhuma encontrada | 0 IDs e 0 registros repetidos |